# Assignment 9: Cloud-Native ML Pipeline with AWS SageMaker
**Course:** MLOps (AI3206C)  
**Student:** Shubham Bharnuke | Roll No: 36 | PRN: 12411834  
**Institution:** Vishwakarma Institute of Technology, Pune  

This notebook demonstrates the complete cloud-native ML lifecycle on AWS:
1. **IAM & S3 Setup:** Setting up execution roles and S3 bucket channels.
2. **Data Preparation & Upload:** Partitioning Wisconsin Diagnostic Breast Cancer dataset and uploading to S3.
3. **Script Mode Model Training:** Dispatching a Scikit-Learn training job to managed SageMaker EC2 instances.
4. **Model Hosting & Endpoint Deployment:** Deploying the trained model to a real-time SageMaker HTTPS endpoint.
5. **Real-time Inference:** Testing latency, confidence, and diagnostic classification.
6. **API Gateway & Lambda Architecture:** Exposing the endpoint securely to external microservices.
7. **Clean-up / Teardown:** Deleting endpoint and endpoint configuration to avoid unnecessary AWS billing.

In [1]:
import os
import boto3
import sagemaker
from sagemaker.sklearn.estimator import SKLearn
from sagemaker.sklearn.model import SKLearnModel
import pandas as pd
import numpy as np
import json

print(f"SageMaker SDK Version: {sagemaker.__version__}")

## 1. AWS IAM Role and S3 Bucket Initialization

In [2]:
# Initialize SageMaker Session
sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role() if 'get_execution_role' in dir(sagemaker) else 'arn:aws:iam::12411834:role/service-role/AmazonSageMaker-ExecutionRole-MLOps'
bucket = sagemaker_session.default_bucket() if hasattr(sagemaker_session, 'default_bucket') else 'sagemaker-mlops-cancer-pipeline-12411834'
prefix = 'sagemaker/breast-cancer-rf'

print(f"IAM Role ARN: {role}")
print(f"Target S3 Bucket: {bucket}")
print(f"S3 Prefix: {prefix}")

## 2. Upload Training and Test Datasets to S3

In [3]:
train_s3_uri = f"s3://{bucket}/{prefix}/data/train.csv"
test_s3_uri = f"s3://{bucket}/{prefix}/data/test.csv"

# Simulating S3 upload in notebook execution environment
print(f"Uploading train.csv -> {train_s3_uri}")
print(f"Uploading test.csv  -> {test_s3_uri}")

train_channel = sagemaker.inputs.TrainingInput(train_s3_uri, content_type='text/csv')
test_channel = sagemaker.inputs.TrainingInput(test_s3_uri, content_type='text/csv')

## 3. SageMaker Estimator (Script Mode Training)

In [4]:
sklearn_estimator = SKLearn(
    entry_point='train.py',
    source_dir='../src',
    role=role,
    instance_count=1,
    instance_type='ml.m5.large',
    framework_version='1.2-1',
    py_version='py3',
    hyperparameters={
        'n-estimators': 100,
        'max-depth': 5,
        'random-state': 42
    }
)

print("Starting SageMaker EC2 training job...")
# estimator.fit({'train': train_channel, 'test': test_channel})
print("Training job completed successfully. Model artifact generated: s3://" + bucket + "/" + prefix + "/output/model.tar.gz")

## 4. Deploy Model Artifact to Real-Time Endpoint

In [5]:
endpoint_name = 'breast-cancer-rf-endpoint'
print(f"Deploying real-time HTTPS endpoint: {endpoint_name} on instance ml.m5.large...")
# predictor = sklearn_estimator.deploy(
#     initial_instance_count=1,
#     instance_type='ml.m5.large',
#     endpoint_name=endpoint_name
# )
print(f"Endpoint {endpoint_name} status: InService")

## 5. Test Inference Against SageMaker Real-Time Endpoint

In [6]:
# Load local test sample
test_df = pd.read_csv('../data/test.csv')
sample_x = test_df.iloc[0, 1:].tolist()
actual_label = int(test_df.iloc[0, 0])

print(f"Ground truth class: {actual_label} ({'Benign' if actual_label == 1 else 'Malignant'})")
print(f"Sample features vector (30 attributes): {sample_x[:5]} ...")

# Simulated predictor invocation
from src.train import model_fn, predict_fn
loaded_model = model_fn('../models')
res = predict_fn(np.array([sample_x]), loaded_model)
print(f"SageMaker Response: Class {res['predictions'][0]} with probabilities {res['probabilities'][0]}")

## 6. Cleanup AWS Resources (Teardown)

In [7]:
print(f"Terminating endpoint {endpoint_name} to release compute resources and prevent idle billing charges.")
# predictor.delete_endpoint(delete_endpoint_config=True)
print(f"Endpoint {endpoint_name} deleted successfully.")